---
title: "动态规划 (DP)——状态机动态规划"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [11]:
#| code-fold: true
from typing import List

## 📝 题目 121：买卖股票的最佳时机

::: {.callout-caution}
### 📖 题目描述
给定一个数组 `prices` ，它的第 `i` 个元素 `prices[i]` 表示一支给定股票第 `i` 天的价格。

你只能选择 **某一天** 买入这只股票，并选择在 **未来的某一个不同的日子** 卖出该股票。设计一个算法来计算你所能获取的最大利润。

返回你可以从这笔交易中获取的最大利润。如果你不能获取任何利润，返回 `0` 。

**输入输出示例**：

* **示例 1**：
    * **输入**：`[7,1,5,3,6,4]`
    * **输出**：`5`
    * **解释**：在第 2 天（股票价格 = 1）的时候买入，在第 5 天（股票价格 = 6）的时候卖出，最大利润 = 6-1 = 5 。
      注意利润不能是 7-1 = 6, 因为卖出价格需要大于买入价格；同时，你不能在买入前卖出股票。

* **示例 2**：
    * **输入**：`prices = [7,6,4,3,1]`
    * **输出**：`0`
    * **解释**：在这种情况下, 没有交易完成, 所以最大利润为 0。
:::

In [4]:
class Solution121:
    def maxProfit(self, prices: List[int]) -> int:
        dp = [[0] * 2 for _ in range(len(prices))]  # 创建 N 行 2 列的二维数组，dp[i][0]表示不持有股票的状态，dp[i][1]表示持有股票的状态
        dp[0][1] = -prices[0]  # 第 0 天，持有股票的状态，现金是负的买入价
        for i in range(1, len(prices)):
            dp[i][0] = max(dp[i - 1][0], dp[i - 1][1] + prices[i])  # 状态 0：今天不持有 = max(昨天也不持有, 昨天持有今天卖了赚回差价)
            dp[i][1] = max(dp[i - 1][1], - prices[i])  # 状态 1：今天持有 = max(昨天就持有一直死扛, 昨天不持有今天刚刚买入)
        return dp[-1][0]

In [5]:
#| code-fold: true
def test_121(func):
    test_cases = [
        {"desc": "示例 1 (常规波动)", "prices": [7, 1, 5, 3, 6, 4], "expected": 5},
        {"desc": "示例 2 (一路暴跌)", "prices": [7, 6, 4, 3, 1], "expected": 0}, # 亏钱就不买，利润为 0
        {"desc": "一路暴涨", "prices": [1, 2, 3, 4, 5], "expected": 4},
        {"desc": "只有两天", "prices": [2, 4], "expected": 2},
        {"desc": "只有一天", "prices": [5], "expected": 0},
        {"desc": "过山车震荡", "prices": [3, 2, 6, 5, 0, 3], "expected": 4},
        {"desc": "深V反弹", "prices": [5, 4, 3, 2, 1, 8], "expected": 7}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["prices"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_121(Solution121().maxProfit)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 5    | 5    | 示例 1 (常规波动)
2    | ✅ PASS | 0    | 0    | 示例 2 (一路暴跌)
3    | ✅ PASS | 4    | 4    | 一路暴涨
4    | ✅ PASS | 2    | 2    | 只有两天
5    | ✅ PASS | 0    | 0    | 只有一天
6    | ✅ PASS | 4    | 4    | 过山车震荡
7    | ✅ PASS | 7    | 7    | 深V反弹
-----------------------------------------------------------------
测试总结: 通过 7/7


::: {.callout-important}
### 💡 思路讲解

每天结束时，你手里只有两种确定的状态：
* **状态 0**：手里**没有**股票（可能是从来没买过，也可能是今天或之前卖掉了）。
* **状态 1**：手里**持有**股票（可能是今天刚买的，也可能是之前买的还在死扛）。

1. **确定 dp 数组以及下标的含义**：
   - 定义一个二维数组 `dp`，大小为 `n x 2`。
   - `dp[i][0]` 表示：第 `i` 天结束时，**不持有**股票时的最大手头现金。
   - `dp[i][1]` 表示：第 `i` 天结束时，**持有**股票时的最大手头现金。
   - *注意：我们默认初始现金为 0，买入股票相当于花钱，现金会变成负数。*

2. **确定递推公式（状态转移方程）**：
   - **如果今天结束时不持有股票 (`dp[i][0]`)**，有两个来源：
     - 昨天就不持有，今天继续观望：`dp[i-1][0]`
     - 昨天持有，今天我把它卖了（赚回了 `prices[i]` 的现金）：`dp[i-1][1] + prices[i]`
     - 取最大值：`dp[i][0] = max(dp[i-1][0], dp[i-1][1] + prices[i])`
   - **如果今天结束时持有股票 (`dp[i][1]`)**，有两个来源：
     - 昨天就持有，今天继续死扛：`dp[i-1][1]`
     - 昨天不持有，今天我**刚刚买入**（注意题目规定**只能买卖一次**，所以只要买入，之前的现金必定是 0，买入后现金直接变成 `-prices[i]`）：`-prices[i]`
     - 取最大值：`dp[i][1] = max(dp[i-1][1], -prices[i])`

3. **dp 数组如何初始化**：
   - 第 0 天（也就是第一天）：
     - 不持有股票：什么都没干，现金为 0。`dp[0][0] = 0`。
     - 持有股票：必须花钱买当天的股票，现金变成负的。`dp[0][1] = -prices[0]`。

4. **确定遍历顺序**：
   - 从第 1 天遍历到最后一天，正序遍历。

5. **最终结果**：
   - 最后一天结束时，肯定是把股票卖掉（不持有）手头现金才最多，所以返回 `dp[-1][0]`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。只需遍历一次 `prices` 数组。
* **空间复杂度**：$O(N)$（如果使用二维 dp 数组）。当然，因为今天只依赖昨天，你完全可以用两个变量来滚动压缩到 $O(1)$！
:::

## 📝 题目 122：买卖股票的最佳时机 II

::: {.callout-caution}
### 📖 题目描述
给你一个整数数组 `prices` ，其中 `prices[i]` 表示某支股票第 `i` 天的价格。

在每一天，你可以决定是否购买和/或出售股票。你在任何时候 **最多** 只能持有 **一股** 股票。你也可以先购买，然后在 **同一天** 出售。

返回 你能获得的 **最大** 利润 。

**输入输出示例**：

* **示例 1**：
    * **输入**：`prices = [7,1,5,3,6,4]`
    * **输出**：`7`
    * **解释**：在第 2 天（股票价格 = 1）的时候买入，在第 3 天（股票价格 = 5）的时候卖出, 这笔交易所能获得利润 = 5 - 1 = 4 。
      随后，在第 4 天（股票价格 = 3）的时候买入，在第 5 天（股票价格 = 6）的时候卖出, 这笔交易所能获得利润 = 6 - 3 = 3 。
      总利润为 4 + 3 = 7 。

* **示例 2**：
    * **输入**：`prices = [1,2,3,4,5]`
    * **输出**：`4`
    * **解释**：在第 1 天（股票价格 = 1）的时候买入，在第 5 天 （股票价格 = 5）的时候卖出, 这笔交易所能获得利润 = 5 - 1 = 4 。
      总利润为 4 。

* **示例 3**：
    * **输入**：`prices = [7,6,4,3,1]`
    * **输出**：`0`
    * **解释**：在这种情况下, 交易无法获得正利润，所以不参与交易可以获得最大利润，最大利润为 0 。
:::

In [6]:
class Solution122:
    def maxProfit(self, prices: List[int]) -> int:
        dp = [[0] * 2 for _ in range(len(prices))]  # 创建 N 行 2 列的二维数组，dp[i][0]表示不持有股票的状态，dp[i][1]表示持有股票的状态
        dp[0][1] = -prices[0]  # 第 0 天，持有股票的状态，现金是负的买入价
        for i in range(1, len(prices)):
            dp[i][0] = max(dp[i - 1][0], dp[i - 1][1] + prices[i])  # 状态 0：今天不持有 = max(昨天也不持有, 昨天持有今天卖了赚回差价)
            dp[i][1] = max(dp[i - 1][1], dp[i - 1][0]- prices[i])  # 状态 1：今天持有 = max(昨天就持有一直死扛, 昨天不持有今天刚刚买入)
        return dp[-1][0]

In [7]:
#| code-fold: true
def test_122(func):
    test_cases = [
        {"desc": "示例 1 (多次波段)", "prices": [7, 1, 5, 3, 6, 4], "expected": 7},
        {"desc": "示例 2 (单边上涨)", "prices": [1, 2, 3, 4, 5], "expected": 4},
        {"desc": "示例 3 (单边下跌)", "prices": [7, 6, 4, 3, 1], "expected": 0},
        {"desc": "深V反复横跳", "prices": [1, 2, 1, 2, 1, 2], "expected": 3},
        {"desc": "只有两天涨", "prices": [2, 4], "expected": 2},
        {"desc": "只有一天", "prices": [5], "expected": 0},
        {"desc": "W型走势", "prices": [5, 2, 3, 1, 6], "expected": 6}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        # 处理空数组的边界情况，为了脚本健壮性稍微加个保护
        if not tc["prices"]:
            actual = 0
        else:
            actual = func(tc["prices"])

        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_122(Solution122().maxProfit)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 7    | 7    | 示例 1 (多次波段)
2    | ✅ PASS | 4    | 4    | 示例 2 (单边上涨)
3    | ✅ PASS | 0    | 0    | 示例 3 (单边下跌)
4    | ✅ PASS | 3    | 3    | 深V反复横跳
5    | ✅ PASS | 2    | 2    | 只有两天涨
6    | ✅ PASS | 0    | 0    | 只有一天
7    | ✅ PASS | 6    | 6    | W型走势
-----------------------------------------------------------------
测试总结: 通过 7/7


::: {.callout-important}
### 💡 思路讲解

既然可以无限次买卖，状态的定义需要变吗？**完全不需要！** 依然是两个状态：
* `dp[i][0]`：第 `i` 天结束时，**不持有**股票时的最大手头现金。
* `dp[i][1]`：第 `i` 天结束时，**持有**股票时的最大手头现金。

**核心魔法：状态转移方程的微调**

1. **今天结束时不持有股票 (`dp[i][0]`)**：
   - 昨天就不持有：`dp[i-1][0]`
   - 昨天持有，今天卖了：`dp[i-1][1] + prices[i]`
   - 取最大值，**和 121 题一模一样！**
   - `dp[i][0] = max(dp[i-1][0], dp[i-1][1] + prices[i])`

2. **今天结束时持有股票 (`dp[i][1]`)（🔥 唯一变化的地方 🔥）**：
   - 昨天就持有，今天死扛：`dp[i-1][1]`
   - **昨天不持有，今天买入**：
     - 在 121 题（只能买一次）中，既然是今天买入，说明之前绝对没有赚过钱，本金是 0，买入后现金直接是 `-prices[i]`。
     - **但在本题中，因为可以无限次买卖，你今天买入股票的钱，不是凭空来的，而是你【昨天不持有股票时口袋里剩下的钱】（也就是之前交易赚来的利润）！**
     - 所以，今天买入后的现金状态应该是：昨天不持有的现金 **减去** 今天的股价。即 `dp[i-1][0] - prices[i]`。
   - 取最大值：
   - **`dp[i][1] = max(dp[i-1][1], dp[i-1][0] - prices[i])`**

**仅仅是把 121 题方程里的 `0` 换成了 `dp[i-1][0]`，你就打通了无限次交易的任督二脉！** 3. **初始化与遍历顺序**：
   - 和上一题完全相同。`dp[0][0] = 0`，`dp[0][1] = -prices[0]`。正序遍历。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。遍历一次数组。
* **空间复杂度**：$O(N)$（二维 dp 数组）。同样可以使用两个变量优化为 $O(1)$。
:::

## 📝 题目 309：买卖股票的最佳时机含冷冻期

::: {.callout-caution}
### 📖 题目描述
给定一个整数数组 `prices`，其中第 `prices[i]` 表示第 `i` 天的股票价格。

设计一个算法计算出最大利润。在满足以下约束条件下，你可以尽可能地完成更多的交易（多次买卖一支股票）:
* 卖出股票后，你无法在第二天买入股票 (即冷冻期为 1 天)。

**注意：** 你不能同时参与多笔交易（你必须在再次购买前出售掉之前的股票）。

**输入输出示例**：

* **示例 1**：
    * **输入**：`prices = [1,2,3,0,2]`
    * **输出**：`3`
    * **解释**：对应的交易状态为: [买入, 卖出, 冷冻期, 买入, 卖出]

* **示例 2**：
    * **输入**：`prices = [1]`
    * **输出**：`0`
:::

In [8]:
class Solution309:
    def maxProfit(self, prices: List[int]) -> int:
        if not prices:  # 防止空数组
            return 0
        dp = [[0] * 3 for _ in range(len(prices))]
        dp[0][0] = -prices[0]
        for i in range(1, len(prices)):
            dp[i][0] = max(dp[i - 1][0], dp[i - 1][2] - prices[i])  # 状态 0：持有股票
            dp[i][1] = dp[i - 1][0] + prices[i]  # 状态 1：今天刚卖出
            dp[i][2] = max(dp[i - 1][2], dp[i - 1][1])  # 状态 2：平静空仓期
        return max(dp[-1][1], dp[-1][2])

In [10]:
#| code-fold: true
def test_309(func):
    test_cases = [
        {"desc": "示例 1 (包含冷冻期)", "prices": [1, 2, 3, 0, 2], "expected": 3},
        {"desc": "示例 2 (只有一天)", "prices": [1], "expected": 0},
        {"desc": "连续上涨", "prices": [1, 2, 3, 4, 5], "expected": 4},
        {"desc": "连续下跌", "prices": [5, 4, 3, 2, 1], "expected": 0},
        {"desc": "震荡诱多", "prices": [6, 1, 3, 2, 4, 7], "expected": 6}, # 1买3卖(赚2)，冷冻过2，4买7卖(赚3)，共5？不对，应该是1买7卖赚6最大！
        {"desc": "多次冷冻", "prices": [1, 4, 2, 7, 1, 5], "expected": 7} # 1买4卖(3)，冷冻，2买7卖(5)，冷冻，1买5卖(4)，没法操作这么快。正确操作是1买4卖(3)，2买7卖(5)，但中间冷冻期冲突。代码会算出正确答案。
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["prices"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_309(Solution309().maxProfit)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 3    | 3    | 示例 1 (包含冷冻期)
2    | ✅ PASS | 0    | 0    | 示例 2 (只有一天)
3    | ✅ PASS | 4    | 4    | 连续上涨
4    | ✅ PASS | 0    | 0    | 连续下跌
5    | ✅ PASS | 6    | 6    | 震荡诱多
6    | ✅ PASS | 7    | 7    | 多次冷冻
-----------------------------------------------------------------
测试总结: 通过 6/6


::: {.callout-important}
### 💡 思路讲解



因为引入了冷冻期，我们手里“不持有股票”的状态变得不再纯粹了。你需要把它劈成两半，这样才能清晰地画出状态流转的箭头。

每天结束时，你必然处于以下 **3 个状态** 之一：
* **状态 0（持有股票）**：手里捏着股票。可能是今天买的，也可能是之前买的一直没卖。
* **状态 1（刚刚卖出，准备冷冻）**：今天我把股票卖了！这就意味着，明天我**必须**强制进入冷冻期。
* **状态 2（平静的空仓期）**：手里没股票，并且今天也没有卖股票（可能是昨天是冷冻期，或者我已经空仓好几天了）。这个状态的特点是：**随时可以买入**。

接下来，我们用魔法连线（状态转移方程）：

1. **今天结束时持有股票 (`dp[i][0]`)**：
   - 昨天就持有，继续死扛：`dp[i-1][0]`
   - 今天刚买入。既然今天能买，昨天必须是“平静的空仓期（状态 2）”。
   - 方程：`dp[i][0] = max(dp[i-1][0], dp[i-1][2] - prices[i])`

2. **今天结束时刚刚卖出 (`dp[i][1]`)**：
   - 这个状态非常纯粹：既然今天“刚刚卖出”，那昨天**必须**持有股票。
   - 方程：`dp[i][1] = dp[i-1][0] + prices[i]`

3. **今天结束时是平静的空仓期 (`dp[i][2]`)**：
   - 昨天也是平静的空仓期，今天继续看戏：`dp[i-1][2]`
   - 昨天是“刚刚卖出（状态 1）”，今天自动解冻，变成了平静的空仓期：`dp[i-1][1]`
   - 方程：`dp[i][2] = max(dp[i-1][2], dp[i-1][1])`

**初始化的基石：**
* 第 0 天持有（状态 0）：`dp[0][0] = -prices[0]`
* 第 0 天刚卖出（状态 1）：第一天不可能卖，所以初始化为 `0`
* 第 0 天平静空仓（状态 2）：本来就没钱没票，初始化为 `0`

**最后的结果：**
最后一天结束时，想要钱最多，肯定不能持有股票（股票得换成钱）。所以最大利润一定在 `dp[-1][1]`（最后一天刚卖出）和 `dp[-1][2]`（最后一天平静空仓）之间产生，取它俩的最大值即可。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。遍历一次数组。
* **空间复杂度**：$O(N)$。使用 $N \times 3$ 的二维 dp 数组。
:::